## 1. Import Libraries

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore Data

In [19]:
# Load the dataset
file_path = 'HAM10000_features_couleurs.xlsx'
df = pd.read_excel(file_path)

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (10015, 21)

Column names: ['image_id', 'dx', 'age', 'sex', 'localization', 'color_1_R', 'color_1_G', 'color_1_B', 'color_1_proportion', 'color_2_R', 'color_2_G', 'color_2_B', 'color_2_proportion', 'color_3_R', 'color_3_G', 'color_3_B', 'color_3_proportion', 'asym_vert_%', 'asym_horiz_%', 'Perimeter', 'Area']

First few rows:


,image_id,dx,age,sex,localization,color_1_R,color_1_G,color_1_B,color_1_proportion,color_2_R,...,color_2_B,color_2_proportion,color_3_R,color_3_G,color_3_B,color_3_proportion,asym_vert_%,asym_horiz_%,Perimeter,Area
0,ISIC_0027419,nomel,80.0,male,scalp,219.179990,177.235439,198.679157,0.457572,184.425591,...,104.949342,0.151712,207.771385,150.836506,157.093755,0.390716,19.105153,17.450459,1251.785930,85333
1,ISIC_0025030,nomel,80.0,male,scalp,226.107693,187.273683,205.974215,0.442245,189.566767,...,107.560725,0.152756,214.179005,158.707311,160.624952,0.404999,35.036812,16.117057,754.488419,32326
2,ISIC_0026769,nomel,80.0,male,scalp,198.104553,122.933030,132.345778,0.168120,224.884205,...,195.015437,0.402795,217.491455,150.013374,164.787046,0.429084,16.548578,15.708324,1114.560533,72121
3,ISIC_0025661,nomel,80.0,male,scalp,227.012822,170.157376,192.372925,0.345771,217.576529,...,158.461661,0.468781,196.976197,117.661687,125.351988,0.185448,19.936487,28.734881,598.215295,21413
4,ISIC_0031633,nomel,75.0,male,ear,215.138856,162.599185,184.575606,0.301135,145.233630,...,102.063107,0.265354,184.497142,129.514243,139.646944,0.433511,24.592457,17.613228,1274.011327,102934


In [20]:
# Check class distribution
print("Class distribution:")
print(df['dx'].value_counts())
print(f"\nClass proportions:")
print(df['dx'].value_counts(normalize=True))

Class distribution:
dx
nomel    8902
mel      1113
Name: count, dtype: int64

Class proportions:
dx
nomel    0.888867
mel      0.111133
Name: proportion, dtype: float64


## 3. Data Preprocessing

In [21]:
# Encode the target variable 'dx'
le = LabelEncoder()
df['dx'] = le.fit_transform(df['dx'])

print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Separate features (X) and target (y)
X = df.drop(['dx', 'image_id'], axis=1)
y = df['dx']

print(f"\nFeature shape: {X.shape}")
print(f"Target shape: {y.shape}")

Label encoding: {'mel': np.int64(0), 'nomel': np.int64(1)}

Feature shape: (10015, 19)
Target shape: (10015,)


In [22]:
# Identify and encode categorical features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical features: {categorical_features}")

X = pd.get_dummies(X, columns=categorical_features, drop_first=True)
print(f"\nFeature shape after encoding: {X.shape}")

Categorical features: ['sex', 'localization']

Feature shape after encoding: (10015, 33)


In [23]:
# Impute missing values
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set class distribution:")
print(pd.Series(y_train).value_counts())

Training set size: (8012, 33)
Test set size: (2003, 33)

Training set class distribution:
dx
1    7122
0     890
Name: count, dtype: int64


In [24]:
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully")

Features scaled successfully


## 4. Feature Selection

In [25]:
# Feature selection to reduce overfitting and improve generalization
k_features = min(15, X_train_scaled.shape[1])
selector = SelectKBest(f_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

print(f"Selected {X_train_selected.shape[1]} best features out of {X_train_scaled.shape[1]}")

# Define cross-validation strategy (used across all models)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Selected 15 best features out of 33


## 5. Hyperparameter Tuning - Random Forest

In [26]:
# Best parameters found from GridSearchCV:
# {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 1, 
#  'min_samples_split': 2, 'n_estimators': 300}
# Best cross-validation F1 score: 0.9414

# Initialize Random Forest with best parameters
best_rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=30,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=300
)

print("Training Random Forest with optimized parameters...")
best_rf_model.fit(X_train_selected, y_train)
print("Random Forest training complete!")

# --- GRID SEARCH (COMMENTED OUT - USE IF YOU WANT TO RE-TUNE) ---
# rf_param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [10, 20, 30, None],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2']
# }
# 
# rf_base = RandomForestClassifier(random_state=42, class_weight='balanced')
# cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# rf_grid_search = GridSearchCV(
#     estimator=rf_base,
#     param_grid=rf_param_grid,
#     cv=cv_strategy,
#     scoring='f1',
#     n_jobs=-1,
#     verbose=1
# )
# 
# print("Training Random Forest with GridSearchCV...")
# rf_grid_search.fit(X_train_selected, y_train)
# best_rf_model = rf_grid_search.best_estimator_
# print(f"\nBest Random Forest parameters: {rf_grid_search.best_params_}")
# print(f"Best cross-validation F1 score: {rf_grid_search.best_score_:.4f}")

Training Random Forest with optimized parameters...
Random Forest training complete!
Random Forest training complete!


## 6. Hyperparameter Tuning - Gradient Boosting

In [27]:
# Best parameters found from GridSearchCV:
# {'learning_rate': 0.01, 'max_depth': 7, 'min_samples_leaf': 4,
#  'min_samples_split': 2, 'n_estimators': 100, 'subsample': 0.9}
# Best cross-validation F1 score: 0.9424

# Initialize Gradient Boosting with best parameters
best_gb_model = GradientBoostingClassifier(
    random_state=42,
    learning_rate=0.01,
    max_depth=7,
    min_samples_leaf=4,
    min_samples_split=2,
    n_estimators=100,
    subsample=0.9
)

print("Training Gradient Boosting with optimized parameters...")
best_gb_model.fit(X_train_selected, y_train)
print("Gradient Boosting training complete!")

# --- GRID SEARCH (COMMENTED OUT - USE IF YOU WANT TO RE-TUNE) ---
# gb_param_grid = {
#     'n_estimators': [100, 200, 300],
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'max_depth': [3, 5, 7],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'subsample': [0.8, 0.9, 1.0]
# }
# 
# gb_base = GradientBoostingClassifier(random_state=42)
# gb_grid_search = GridSearchCV(
#     estimator=gb_base,
#     param_grid=gb_param_grid,
#     cv=cv_strategy,
#     scoring='f1',
#     n_jobs=-1,
#     verbose=1
# )
# 
# print("Training Gradient Boosting with GridSearchCV (this will take some time)...")
# gb_grid_search.fit(X_train_selected, y_train)
# best_gb_model = gb_grid_search.best_estimator_
# print(f"\nBest Gradient Boosting parameters: {gb_grid_search.best_params_}")
# print(f"Best cross-validation F1 score: {gb_grid_search.best_score_:.4f}")

Training Gradient Boosting with optimized parameters...
Gradient Boosting training complete!
Gradient Boosting training complete!


## 7. Hyperparameter Tuning - SVM

In [28]:
# Best parameters found from GridSearchCV:
# {'C': 0.1, 'gamma': 0.01, 'kernel': 'poly'}
# Best cross-validation F1 score: 0.9399

# Initialize SVM with best parameters
best_svm_model = SVC(
    random_state=42,
    class_weight='balanced',
    probability=True,
    C=0.1,
    gamma=0.01,
    kernel='poly'
)

print("Training SVM with optimized parameters...")
best_svm_model.fit(X_train_selected, y_train)
print("SVM training complete!")

# --- GRID SEARCH (COMMENTED OUT - USE IF YOU WANT TO RE-TUNE) ---
# svm_param_grid = {
#     'C': [0.1, 1, 10, 100],
#     'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
#     'kernel': ['rbf', 'poly']
# }
# 
# svm_base = SVC(random_state=42, class_weight='balanced', probability=True)
# svm_grid_search = GridSearchCV(
#     estimator=svm_base,
#     param_grid=svm_param_grid,
#     cv=cv_strategy,
#     scoring='f1',
#     n_jobs=-1,
#     verbose=1
# )
# 
# print("Training SVM with GridSearchCV...")
# svm_grid_search.fit(X_train_selected, y_train)
# best_svm_model = svm_grid_search.best_estimator_
# print(f"\nBest SVM parameters: {svm_grid_search.best_params_}")
# print(f"Best cross-validation F1 score: {svm_grid_search.best_score_:.4f}")

Training SVM with optimized parameters...
SVM training complete!
SVM training complete!


## 8. Hyperparameter Tuning - Logistic Regression

In [29]:
# Best parameters found from GridSearchCV:
# {'C': 10, 'penalty': 'l1', 'solver': 'saga'}
# Best cross-validation F1 score: 0.8185

# Initialize Logistic Regression with best parameters
best_lr_model = LogisticRegression(
    random_state=42,
    class_weight='balanced',
    max_iter=1000,
    C=10,
    penalty='l1',
    solver='saga'
)

print("Training Logistic Regression with optimized parameters...")
best_lr_model.fit(X_train_selected, y_train)
print("Logistic Regression training complete!")

# --- GRID SEARCH (COMMENTED OUT - USE IF YOU WANT TO RE-TUNE) ---
# lr_param_grid = {
#     'C': [0.001, 0.01, 0.1, 1, 10, 100],
#     'penalty': ['l1', 'l2'],
#     'solver': ['liblinear', 'saga']
# }
# 
# lr_base = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
# lr_grid_search = GridSearchCV(
#     estimator=lr_base,
#     param_grid=lr_param_grid,
#     cv=cv_strategy,
#     scoring='f1',
#     n_jobs=-1,
#     verbose=1
# )
# 
# print("Training Logistic Regression with GridSearchCV...")
# lr_grid_search.fit(X_train_selected, y_train)
# best_lr_model = lr_grid_search.best_estimator_
# print(f"\nBest Logistic Regression parameters: {lr_grid_search.best_params_}")
# print(f"Best cross-validation F1 score: {lr_grid_search.best_score_:.4f}")

Training Logistic Regression with optimized parameters...
Logistic Regression training complete!
Logistic Regression training complete!


## 9. Model Comparison on Test Set

In [30]:
# Compare all tuned models
models = {
    'Tuned Random Forest': best_rf_model,
    'Tuned Gradient Boosting': best_gb_model,
    'Tuned Logistic Regression': best_lr_model,
    'Tuned SVM': best_svm_model
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test_selected)
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1_sc = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1_sc,
        'ROC-AUC': auc
    })

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("MODEL COMPARISON ON TEST SET")
print("="*80)
print(results_df.to_string(index=False))

# Find best model based on F1 score
best_model_name = results_df.loc[results_df['F1'].idxmax(), 'Model']
best_f1 = results_df['F1'].max()
print(f"\nBest performing model: {best_model_name} (F1 Score: {best_f1:.4f})")


MODEL COMPARISON ON TEST SET
                    Model  Accuracy  Precision   Recall       F1  ROC-AUC
      Tuned Random Forest  0.888168   0.891742 0.994944 0.940520 0.815897
  Tuned Gradient Boosting  0.889166   0.890672 0.997753 0.941176 0.801418
Tuned Logistic Regression  0.727908   0.961855 0.722472 0.825152 0.814944
                Tuned SVM  0.885671   0.888722 0.996067 0.939338 0.787588

Best performing model: Tuned Gradient Boosting (F1 Score: 0.9412)


## 10. Detailed Evaluation of Best Model

In [31]:
# Get the best model
best_idx = results_df['F1'].idxmax()
best_model = list(models.values())[best_idx]

# Make predictions
y_pred = best_model.predict(X_test_selected)

print("\n" + "="*80)
print(f"DETAILED CLASSIFICATION REPORT - {best_model_name}")
print("="*80)
print(classification_report(y_test, y_pred, target_names=le.classes_))


DETAILED CLASSIFICATION REPORT - Tuned Gradient Boosting
              precision    recall  f1-score   support

         mel       0.56      0.02      0.04       223
       nomel       0.89      1.00      0.94      1780

    accuracy                           0.89      2003
   macro avg       0.72      0.51      0.49      2003
weighted avg       0.85      0.89      0.84      2003



## 11. Cross-Validation Analysis

In [32]:
# Prepare full dataset for cross-validation
X_full_scaled = scaler.fit_transform(X)
X_full_selected = selector.fit_transform(X_full_scaled, y)

# Cross-validate all models
print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS (5-Fold Stratified)")
print("="*80)

cv_results = []
for name, model in models.items():
    cv_acc_scores = cross_val_score(model, X_full_selected, y, cv=cv_strategy, scoring='accuracy')
    cv_f1_scores = cross_val_score(model, X_full_selected, y, cv=cv_strategy, scoring='f1')
    
    cv_results.append({
        'Model': name,
        'CV Accuracy': f"{np.mean(cv_acc_scores):.4f} (+/- {np.std(cv_acc_scores):.4f})",
        'CV F1 Score': f"{np.mean(cv_f1_scores):.4f} (+/- {np.std(cv_f1_scores):.4f})"
    })

cv_results_df = pd.DataFrame(cv_results)
print(cv_results_df.to_string(index=False))


CROSS-VALIDATION RESULTS (5-Fold Stratified)
                    Model         CV Accuracy         CV F1 Score
      Tuned Random Forest 0.8908 (+/- 0.0021) 0.9420 (+/- 0.0011)
  Tuned Gradient Boosting 0.8911 (+/- 0.0014) 0.9422 (+/- 0.0007)
Tuned Logistic Regression 0.7205 (+/- 0.0084) 0.8199 (+/- 0.0062)
                Tuned SVM 0.8871 (+/- 0.0022) 0.9400 (+/- 0.0012)
                    Model         CV Accuracy         CV F1 Score
      Tuned Random Forest 0.8908 (+/- 0.0021) 0.9420 (+/- 0.0011)
  Tuned Gradient Boosting 0.8911 (+/- 0.0014) 0.9422 (+/- 0.0007)
Tuned Logistic Regression 0.7205 (+/- 0.0084) 0.8199 (+/- 0.0062)
                Tuned SVM 0.8871 (+/- 0.0022) 0.9400 (+/- 0.0012)


## 12. Summary and Recommendations

In [33]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nBest Model: {best_model_name}")
print(f"Test F1 Score: {best_f1:.4f}")
print(f"Test Accuracy: {results_df.loc[best_idx, 'Accuracy']:.4f}")
print(f"Test Recall: {results_df.loc[best_idx, 'Recall']:.4f}")
print(f"Test Precision: {results_df.loc[best_idx, 'Precision']:.4f}")
print(f"Test ROC-AUC: {results_df.loc[best_idx, 'ROC-AUC']:.4f}")
print("\nRecommendation: For melanoma detection, high recall is critical to avoid missing cases.")
print("The model with the best balance of recall and precision should be prioritized.")


SUMMARY

Best Model: Tuned Gradient Boosting
Test F1 Score: 0.9412
Test Accuracy: 0.8892
Test Recall: 0.9978
Test Precision: 0.8907
Test ROC-AUC: 0.8014

Recommendation: For melanoma detection, high recall is critical to avoid missing cases.
The model with the best balance of recall and precision should be prioritized.
